# Week 1 -Churn Prediction Pipeline
## Day2 - Preprocessing, SMOTE and Model Training

**Goal:** Clean data. encode features. balance classes. train 3 models. cross validate

**Input:** WA_Fn-UseC_-Telco-Customer-Churn.csv

**Output:** Trained XGBoost model saved as model.pkl

**Author:** Martin James |github.com/M20Jay

## Section 1 - Import Libraries
We import all tools needed for preprocessing, balancing and model training

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings("ignore")

## Section 2- Load and Prepare Data
We load dataset from Day 1 and apply initial cleaning before preprocessing

In [16]:
# Load and Prepare Data
df= pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
# Fix TotalCharges — convert from object to numeric
df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors= 'coerce')

# Drop rows with missing TotalCharges (only 11 rows)
print("Missing TotalCharges:", df['TotalCharges'].isnull().sum())
df.dropna(subset=['TotalCharges'],inplace =True)

#Encode target column with  - Yes = 1 , No =0
df['Churn']=df['Churn'].map({'Yes':1, 'No':0})

print("\nShape after cleaning:", df.shape)
print("\nChurn value counts:")
print(df['Churn'].value_counts())

Missing TotalCharges: 11

Shape after cleaning: (7032, 21)

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64


## Section 3 — Feature Engineering
We recreate the 3 features from Day 1 and drop columns not needed for modelling.

In [ ]:
#  Feature Engineering

# Recreate features from Day 1
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 72],labels=['New', 'Mid', 'Loyal'])

df['charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)

df['is_high_value'] = (df['MonthlyCharges'] > 70).astype(int)

# Drop columns not needed for modelling
df.drop(columns=['customerID'], inplace =True)
print("Features after engineering:")
print(df.columns.to_list())
print("\nShape :", df.shape)


Features after engineering:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'tenure_group', 'charges_per_month', 'is_high_value']

Shape : (7032, 23)


## Section 4 -Encode Categorical Variables
We convert all text columns to numbers so the model can process them

In [ ]:
# Ecode categorical variables
le =LabelEncoder()

# Columns to encode
df.select_dtypes(include='object').columns.tolist()

# Columns to encode
cat_cols =['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies','Contract', 'PaperlessBilling','PaymentMethod']


for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Convert tenure_group from category to string for encoding
df['tenure_group'] = df['tenure_group'].astype(str)

print("Encoding complete ✅")
print(df.dtypes)


Encoding complete ✅
gender                  int64
SeniorCitizen           int64
Partner                 int64
Dependents              int64
tenure                  int64
PhoneService            int64
MultipleLines           int64
InternetService         int64
OnlineSecurity          int64
OnlineBackup            int64
DeviceProtection        int64
TechSupport             int64
StreamingTV             int64
StreamingMovies         int64
Contract                int64
PaperlessBilling        int64
PaymentMethod           int64
MonthlyCharges        float64
TotalCharges          float64
Churn                   int64
tenure_group         category
charges_per_month     float64
is_high_value           int64
dtype: object
